In [21]:
%pip install -r requirements.txt transformers

  Using cached huggingface_hub-2.4.0-py3-none-any.whl.metadata (16 kB)
  Using cached tokenizers-0.23.3-cp310-abi3-macosx_11_0_arm64.whl.metadata (9.8 kB)
  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached hf_xet-1.8.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached httpx2-2.16.0-py3-none-any.whl.metadata (10 kB)
  Using cached anyio-4.16.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached httpcore2-2.16.0-py3-none-any.whl.metadata (27 kB)
  Using cached truststore-0.10.5-py3-none-any.whl.metadata (4.5 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 12.0 MB/s  0:00:01 eta 0:00:01
Using cached huggingface_hub-2.4.0-py3-none-any.whl (857 kB)
Using cached hf_xe

In [37]:
import pandas as pd

df = pd.read_csv("movies-with-sentiments.csv")

df_pkl = df.to_pickle("movies-with-sentiments.pkl")

In [22]:
import pandas as pd
from transformers import pipeline

# Load sentiment analysis pipeline (runs on GPU if available via device=0)
sentiment_analyzer = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=-1 # Set to 0 if you have a CUDA-compatible GPU
)

def extract_sentiment_features(review_list):
    """
    Takes a list of review text strings, computes sentiment scores, 
    and returns aggregated numeric features.
    """
    if not review_list or not isinstance(review_list, list):
        return {
            "mean_sentiment": 0.0,
            "sentiment_std": 0.0,
            "positive_ratio": 0.0
        }

    # Truncate reviews to 512 tokens max to prevent transformer overflow errors
    truncated_reviews = [rev[:512] for rev in review_list if isinstance(rev, str) and rev.strip()]
    
    if not truncated_reviews:
        return {"mean_sentiment": 0.0, "sentiment_std": 0.0, "positive_ratio": 0.0}

    try:
        # Batch inference
        results = sentiment_analyzer(truncated_reviews, truncation=True, max_length=512)
    except Exception as e:
        print(f"Error during sentiment batch inference: {e}")
        return {"mean_sentiment": 0.0, "sentiment_std": 0.0, "positive_ratio": 0.0}

    # Map labels to numeric weights (e.g., Negative: 0, Neutral: 0.5, Positive: 1.0)
    label_mapping = {"negative": 0.0, "neutral": 0.5, "positive": 1.0}
    
    scores = [label_mapping.get(res['label'].lower(), 0.5) * res['score'] for res in results]
    
    series = pd.Series(scores)
    
    return {
        "mean_sentiment": series.mean(),
        "sentiment_std": series.std() if len(series) > 1 else 0.0,
        "positive_ratio": (series > 0.7).mean()
    }



/Users/nourallam/ml-project-scripts/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1112 14:25:06.847000 20415 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 35532.96it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
print(df['Reviews'].head())
sentiment_features = df['Reviews'].head().apply(extract_sentiment_features)
print(sentiment_features[0])

0    [Can’t say it’s bad, not even close, but it’s ...
1    [masterpiece, Grandiose, +90000000000000000000...
2    [Willlllssssoooooonnnnnn, WILSON! LO SIENTO, A...
3    [He sucks, and this concept is so intrusive......
4    [zini was the first incel, This is the only mo...
Name: Reviews, dtype: object
{'mean_sentiment': np.float64(0.18762515981992087), 'sentiment_std': np.float64(0.2557647832786662), 'positive_ratio': np.float64(0.06666666666666667)}
